In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("../data/telco_churn_raw.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Drop customerID — it's just an identifier, not a feature
df = df.drop('customerID', axis=1)

print(df.shape)

(7032, 20)


In [2]:
X = df.drop('Churn', axis=1)   # everything except the target
y = df['Churn']                 # what we're predicting

print(f"Features: {X.shape[1]}")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")

Features: 19
Target distribution:
Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64


In [3]:
# One-hot encoding: create a binary column for each category
X = pd.get_dummies(X, drop_first=True)

print(f"Features after encoding: {X.shape[1]}")

Features after encoding: 30


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% held out for testing
    random_state=42,     # reproducible split
    stratify=y           # keep the same churn ratio in both sets
)

In [5]:
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaling to test data
X_test_scaled = scaler.transform(X_test)

In [6]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = model.predict(X_test_scaled)

In [7]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

Accuracy: 0.8038

=== Classification Report ===
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1033
       Churn       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



In [8]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"                  Predicted No    Predicted Yes")
print(f"Actual No         {cm[0,0]:<15} {cm[0,1]}")
print(f"Actual Yes        {cm[1,0]:<15} {cm[1,1]}")

Confusion Matrix:
                  Predicted No    Predicted Yes
Actual No         916             117
Actual Yes        159             215


In [9]:
import pickle

with open('../models/baseline_logistic.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Baseline model saved.")

Baseline model saved.
